<a href="https://colab.research.google.com/github/dtoralg/TheValley_MDS/blob/main/%5B08%5D%20-%20Ingenieria_de_Variables_II/%5B01%5D%20-%20Notebooks/E2_Reduce_y_Modela.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E2 · Reduce y modela - Ingeniería de Variables II

## Introducción

Ya sabemos reducir dimensiones con PCA. La pregunta práctica es: **¿compensa?** Vamos a
entrenar el mismo modelo (una regresión logística) de dos formas y a compararlas de forma
justa:

1. **Sin PCA**: escalar y entrenar con **todas** las columnas.
2. **Con PCA**: escalar, reducir con PCA y entrenar con **pocas** componentes.

Compararemos **AUC** (acierto), **número de columnas** y **tiempo de entrenamiento**. Y todo
dentro de un **Pipeline**, con el `fit` solo en train para no tener fugas.

## Objetivos del ejercicio

- Montar un **Pipeline** `escalar -> PCA -> modelo`.
- Comparar **AUC y tiempo** con y sin reducción de dimensionalidad.
- Entender cuándo **compensa** reducir (menos columnas, modelo más simple y rápido).
- Hacer PCA **sin fugas** (ajustando solo con train).

## Descripción del dataset (sensores, dataset "ancho")

Para esta sesión usamos un dataset **sintético y reproducible** que imita un caso muy
común: **muchísimas columnas pero pocas dimensiones reales**. Piensa en cientos de
sensores que, en el fondo, miden unas pocas cosas (temperatura, presión, vibración...).

Lo generamos dentro del propio notebook con `generar_datos_anchos`, así que es
autocontenido en Colab. Por dentro:

- Hay unos pocos **factores latentes** (la "verdad" oculta) que generan la señal.
- Cada **columna `sensor_XXX`** es una mezcla de esos factores más algo de ruido, así que
  muchas columnas **dicen casi lo mismo** (están muy correlacionadas).
- Las columnas tienen **escalas muy distintas** a propósito (de 1 a 1000), para ver por
  qué hay que escalar antes de PCA.
- `target` es la etiqueta (la clase de cada fila).

> La idea de fondo de la clase: *más columnas no significa más información*. Aquí lo vemos
> en directo, porque la información real vive en muy pocas dimensiones.

### 1. Importar librerías necesarias

In [ ]:
import numpy as np
import pandas as pd
from time import perf_counter
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

### 2. Un dataset MUY ancho y su partición train/test

In [ ]:
import numpy as np
import pandas as pd

def generar_datos_anchos(n=800, n_cols=60, n_latentes=5, n_clases=2,
                         separacion=2.5, ruido=0.6, semilla=42):
    # Genera un dataset "ancho": muchas columnas (sensores) pero POCAS dimensiones
    # reales. Unos pocos factores latentes generan casi toda la informacion; el
    # resto de columnas son mezclas de esos factores mas ruido. Asi PCA puede
    # recuperar la estructura con pocas componentes.
    rng = np.random.default_rng(semilla)

    # Centros de cada clase en el espacio latente (grupos separados)
    centros = rng.normal(scale=separacion, size=(n_clases, n_latentes))
    y = rng.integers(0, n_clases, size=n)
    Z = centros[y] + rng.normal(size=(n, n_latentes))      # factor latente de cada fila

    # Cada columna observada = mezcla ponderada de los factores latentes + ruido
    cargas = rng.normal(size=(n_latentes, n_cols))
    X = Z @ cargas + ruido * rng.normal(size=(n, n_cols))

    # Escalas MUY distintas por columna (para motivar StandardScaler antes de PCA)
    escalas = rng.uniform(1, 1000, size=n_cols)
    X = X * escalas

    cols = [f"sensor_{i:03d}" for i in range(n_cols)]
    df = pd.DataFrame(X, columns=cols)
    df["target"] = y
    return df

In [ ]:
# 4000 columnas, pero la información real vive en solo 8 dimensiones latentes
df = generar_datos_anchos(n=3000, n_cols=4000, n_latentes=8,
                          separacion=1.4, semilla=7)
X = df.drop(columns="target")
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y)

print("Columnas (dimensiones):", X.shape[1])
print("Train:", X_train.shape, "| Test:", X_test.shape)

### 3. Modelo SIN PCA (todas las columnas)

In [ ]:
modelo_sin = Pipeline([
    ("escalar", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000)),
])

t0 = perf_counter()
modelo_sin.fit(X_train, y_train)
t_sin = perf_counter() - t0

auc_sin = roc_auc_score(y_test, modelo_sin.predict_proba(X_test)[:, 1])
print(f"Sin PCA  -> columnas: {X_train.shape[1]:4d} | tiempo fit: {t_sin:.3f}s | AUC test: {auc_sin:.3f}")

### 4. Modelo CON PCA (escalar -> PCA -> modelo)

In [ ]:
modelo_pca = Pipeline([
    ("escalar", StandardScaler()),
    # 12 componentes (algo por encima de las 8 dimensiones reales); 'randomized' es rápido
    ("pca", PCA(n_components=12, svd_solver="randomized", random_state=0)),
    ("clf", LogisticRegression(max_iter=2000)),
])

t0 = perf_counter()
modelo_pca.fit(X_train, y_train)
t_pca = perf_counter() - t0

n_comp = modelo_pca.named_steps["pca"].n_components_
var_ret = modelo_pca.named_steps["pca"].explained_variance_ratio_.sum()
auc_pca = roc_auc_score(y_test, modelo_pca.predict_proba(X_test)[:, 1])
print(f"Con PCA  -> columnas: {n_comp:4d} | varianza conservada: {var_ret*100:.0f}% "
      f"| tiempo fit: {t_pca:.3f}s | AUC test: {auc_pca:.3f}")

### 5. Comparativa

In [ ]:
tabla = pd.DataFrame({
    "modelo": ["Sin PCA", "Con PCA"],
    "n_columnas": [X_train.shape[1], n_comp],
    "tiempo_fit_s": [round(t_sin, 3), round(t_pca, 3)],
    "AUC_test": [round(auc_sin, 3), round(auc_pca, 3)],
})
print(tabla.to_string(index=False))
print(f"\nReducción de columnas: {X_train.shape[1]} -> {n_comp}")
print(f"El modelo con PCA entrena en torno a {t_sin/t_pca:.1f}x más rápido con un AUC parecido.")

### 6. ¿Compensa?

En un caso así (muchas columnas redundantes) con PCA conseguimos **un AUC muy parecido usando
muchísimas menos columnas**. Eso significa un modelo **más simple, más rápido de entrenar y
evaluar, y más fácil de mantener**. Si el AUC se mantiene, la reducción compensa casi siempre.

Y muy importante: el PCA va **dentro del Pipeline**, así que en cada `fit` se aprende **solo
con train**. Eso evita la fuga de datos (lo vemos a fondo en E4).

### Reflexión

1. ¿Cuántas columnas hemos necesitado con PCA para mantener el AUC? ¿Por qué tan pocas?
2. ¿Por qué el ahorro de tiempo es grande aquí y sería pequeño con pocas columnas?
3. ¿Qué pasaría con el AUC si redujéramos demasiado (por ejemplo, a 2 componentes)?
4. ¿Por qué es importante que el PCA esté dentro del Pipeline y no antes?